# 03 Feature Engineering
Create and validate features for downstream modeling.

In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

PROCESSED_DIR = ROOT / "data" / "processed"

In [2]:
orders = pd.read_parquet(PROCESSED_DIR / "olist_orders_clean.parquet")
items = pd.read_parquet(PROCESSED_DIR / "olist_order_items_clean.parquet")
customers = pd.read_parquet(PROCESSED_DIR / "olist_customers_clean.parquet")
payments = pd.read_parquet(PROCESSED_DIR / "olist_order_payments_clean.parquet")

In [3]:
order_value = (
    items.groupby("order_id", as_index=False)
    .agg(
        item_count=("order_item_id", "count"),
        total_price=("price", "sum"),
        total_freight=("freight_value", "sum"),
    )
)

order_features = (
    orders.merge(order_value, on="order_id", how="left")
    .merge(payments.groupby("order_id", as_index=False).agg(total_payment=("payment_value", "sum")), on="order_id", how="left")
)

customer_features = (
    order_features.merge(customers[["customer_id", "customer_unique_id", "customer_state"]], on="customer_id", how="left")
    .groupby("customer_unique_id", as_index=False)
    .agg(
        total_orders=("order_id", "nunique"),
        total_spend=("total_payment", "sum"),
        avg_order_value=("total_payment", "mean"),
    )
)

display(order_features.head())
display(customer_features.head())

order_features.to_parquet(PROCESSED_DIR / "feature_mart_order_level.parquet", index=False)
customer_features.to_parquet(PROCESSED_DIR / "feature_mart_customer_level.parquet", index=False)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,temporal_violation,item_count,total_price,total_freight,total_payment
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,False,1.0,29.99,8.72,38.71
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,False,1.0,118.70,22.76,141.46
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,False,1.0,159.90,19.22,179.12
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,False,1.0,45.00,27.20,72.20
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,False,1.0,19.90,8.72,28.62


,customer_unique_id,total_orders,total_spend,avg_order_value
0,0000366f3b9a7992bf8c76cfdf3221e2,1,141.90,141.90
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1,27.19,27.19
2,0000f46a3911fa3c0805444483337064,1,86.22,86.22
3,0000f6ccb0745a6a4b88665a16c9f078,1,43.62,43.62
4,0004aac84e0df4da2b147fca70cf8255,1,196.89,196.89
